### 1. wrap_model_call

In [8]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

out_model = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

1.1 基于装饰器实现

In [2]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from typing import Callable, Any


@wrap_model_call
def wrap_model_call_middleware(
        request: ModelRequest, # 包含即将发送给大模型的所有请求数据（如消息列表、温度等）
        handler: Callable[[ModelRequest], ModelResponse], # 核心句柄：代表下一个中间件或最终的大模型调用服务
) -> ModelResponse | None:
    # 动态纂改用户发出的最后一条消息的内容，悄悄往里面追加字符串
    # 典型应用：统一再底层为所有请求追求特殊的 Prompt 提示词（例如：“请用中文回答”、“禁止透露公司机密”等）。
    request.messages[-1].content += " ---> wrap_model_call_before <--- "
    # 将修改后的请求传递给handler，真正去调用大模型（或者流转到下一个拦截器）
    # 这一步会产生真实的 Token 消耗并等待大模型响应
    response = handler(request)
    # 大模型返回响应后，在将想用交付给 Agent 状态机之前，对其内容进行直接纂改
    # response.result 是一个消息列表，修改其第一条返回消息的内容
    # 典型应用：做底层的文本敏感词过滤、输出格式强行格式化、或是统一添加某些后处理标记
    response.result[0].content += " ---> wrap_model_call_after <--- "
    # 将修改完的响应体返回，继续维持 Agent 生命周期流转
    return response

agent = create_agent(
    model=out_model,
    middleware=[wrap_model_call_middleware],
)

resp = agent.invoke({
    "messages": [HumanMessage("你好啊")],
})

for msg in resp["messages"]:
    msg.pretty_print()


================================ Human Message =================================

你好啊 ---> wrap_model_call_before <--- 
================================== Ai Message ==================================

你好！👋 看到你提到了 `wrap_model_call_before`，这通常是推理服务、Agent 框架（如 LangChain、LlamaIndex）或自研网关中用于**前置拦截/预处理**的标准钩子（Hook）。常见用途包括：
- 参数校验 & Prompt 安全过滤
- 缓存命中检测 / 限流控制
- 日志埋点 & 审计追踪
- Token 计费或配额预扣

⚠️ 注意：我本身是云端纯对话模型，**不直接暴露或运行你本地的 wrapper 逻辑**。如果你是在自己的服务里实现这个钩子，我可以帮你：
1. 提供多语言（Python/C++/Go）的安全 Wrapper 模板
2. 排查调用中断、超时或 Hook 死锁问题
3. 对接 vLLM / Triton / FastChat / LangChain 的最佳实践

需要我针对你的具体框架或报错细节给出一段可运行的示例吗？随时把上下文贴出来～ ---> wrap_model_call_after <---


1.2 基于类实现

In [4]:
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from typing import Callable

class WrapModelCallMiddleware(AgentMiddleware):
    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse | None:
        request.messages[-1].content += " ---> wrap_model_call_before <--- "
        response = handler(request)
        response.result[0].content += " ---> wrap_model_call_after <--- "

        return response

agent = create_agent(
    model=out_model,
    middleware=[WrapModelCallMiddleware()],
)

resp = agent.invoke({
    "messages": [HumanMessage("你好啊")],
})

for msg in resp["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好啊 ---> wrap_model_call_before <--- 
================================== Ai Message ==================================

你好！很高兴与你交流。后面的 `wrap_model_call_before` 看起来像是某种调用钩子或日志标记，如果是调试、测试或有具体的技术问题，随时告诉我具体场景～ 

有什么我可以帮你的吗？ 😊 ---> wrap_model_call_after <---


使用场景：用于拦截、重试、缓存模型调用。

场景1：重试逻辑

In [9]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from typing import Callable
import time
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

out_model = init_chat_model(
    model="qwen3.7-flash2",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

@wrap_model_call
def retry_model(
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        """自动重试失败的模型调用"""
        max_retries = 3

        for attempt in range(max_retries):
            try:
                print(f"🔄 尝试调用模型（第 {attempt + 1}/{max_retries} 次）")
                return handler(request)
            except Exception as e:
                if attempt == max_retries - 1:
                    print(f"❌ 所有重试失败：{e}")
                    raise
                # 指数退避
                wait_time = 2 ** attempt
                print(f"⚠ 调用失败：{e}，{wait_time} 秒后重试")
                time.sleep(wait_time)

agent = create_agent(
    model=out_model,
    middleware=[retry_model],
)

resp = agent.invoke({
    "messages": [HumanMessage("你好啊")],
})

for msg in resp["messages"]:
    msg.pretty_print()

🔄 尝试调用模型（第 1/3 次）
⚠ 调用失败：Error code: 404 - {'error': {'message': 'Model not exist.', 'id': 'f9263aed-2e66-9919-a0ff-73b5638e94b3', 'type': 'invalid_request_error', 'code': 'model_not_found'}}，1 秒后重试
🔄 尝试调用模型（第 2/3 次）
⚠ 调用失败：Error code: 404 - {'error': {'message': 'Model not exist.', 'id': '53afc0ca-22d7-9e75-87a5-1bac41f18b9c', 'type': 'invalid_request_error', 'code': 'model_not_found'}}，2 秒后重试
🔄 尝试调用模型（第 3/3 次）
❌ 所有重试失败：Error code: 404 - {'error': {'message': 'Model not exist.', 'id': '82659bca-d3b6-97a8-b43e-9cdee431861a', 'type': 'invalid_request_error', 'code': 'model_not_found'}}


OpenAIModelNotFoundError: Error code: 404 - {'error': {'message': 'Model not exist.', 'id': '82659bca-d3b6-97a8-b43e-9cdee431861a', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

场景2：响应缓存

In [21]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.tools import tool
from typing import Callable
import hashlib
import json
import os

load_dotenv(override=True)

# ====================== 1. 初始化模型 ======================
# 使用通义千问 OpenAI 兼容模式接入
out_model = init_chat_model(
    model="qwen-plus",  # 可替换为 qwen-max / qwen-turbo / deepseek-chat 等
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL", "https://dashscope.aliyuncs.com/compatible-mode/v1"),
    temperature=0,  # 温度设为0，相同输入输出更稳定，缓存效果更好
)


# ====================== 2. 定义一个简单工具（可选，演示Agent能力） ======================
@tool
def calculator(expression: str) -> str:
    """计算数学表达式，例如输入 '1+2*3' 返回计算结果"""
    try:
        result = eval(expression)
        return f"计算结果：{result}"
    except Exception as e:
        return f"计算错误：{str(e)}"


# ====================== 3. 模型缓存中间件 ======================
class ModelCache:
    """
    模型响应内存缓存
    扩展功能：
    - 统计缓存命中率
    - 统计调用耗时
    - 支持查看缓存大小
    """
    def __init__(self):
        self.cache = {}
        self.hit_count = 0
        self.miss_count = 0
        self.total_time_saved = 0.0

    def _generate_cache_key(self, request: ModelRequest) -> str:
        """生成唯一缓存键：包含消息内容、系统提示、模型参数"""
        key_data = {
            "messages": [
                {"role": m.type, "content": str(m.content)}
                for m in request.messages
            ],
            "system_prompt": str(request.system_message.content) if request.system_message else "",
            "model": str(request.model),
            "temperature": getattr(request, "temperature", 0),
        }
        return hashlib.md5(
            json.dumps(key_data, sort_keys=True, ensure_ascii=False).encode("utf-8")
        ).hexdigest()

    def create_hook(self):
        @wrap_model_call
        def cache_model(
                request: ModelRequest,
                handler: Callable[[ModelRequest], ModelResponse],
        ) -> ModelResponse:
            cache_key = self._generate_cache_key(request)

            # 命中缓存：直接返回
            if cache_key in self.cache:
                self.hit_count += 1
                hit_time = time.time()
                print(f"\n💾 缓存命中！累计命中 {self.hit_count} 次")
                return self.cache[cache_key]

            # 未命中：调用模型
            self.miss_count += 1
            start_time = time.time()
            print(f"\n🔍 缓存未命中，正在调用模型... (第 {self.miss_count} 次真实调用)")

            response = handler(request)

            elapsed = time.time() - start_time
            self.total_time_saved += elapsed
            print(f"✅ 模型调用完成，耗时：{elapsed:.2f}秒")

            # 存入缓存
            self.cache[cache_key] = response
            return response

        return cache_model

    def get_stats(self):
        """返回缓存统计信息"""
        total = self.hit_count + self.miss_count
        hit_rate = (self.hit_count / total * 100) if total > 0 else 0
        return {
            "缓存条目数": len(self.cache),
            "总调用次数": total,
            "缓存命中次数": self.hit_count,
            "真实调用次数": self.miss_count,
            "缓存命中率": f"{hit_rate:.1f}%",
            "累计节省时间": f"{self.total_time_saved:.2f}秒"
        }


# ====================== 4. 构建带缓存的Agent ======================
cache = ModelCache()

agent = create_agent(
    model=out_model,
    tools=[calculator],  # 可以注释掉tools，纯对话测试
    middleware=[cache.create_hook()],
    system_prompt="你是一个 helpful 的AI助手，回答简洁明了。需要计算时使用calculator工具。"
)


# ====================== 5. 测试Demo ======================
def run_demo():
    print("=" * 60)
    print("LangChain Agent 缓存功能测试Demo")
    print("=" * 60)

    # 测试问题列表（包含重复问题，用于验证缓存命中）
    test_questions = [
        "你好，请介绍一下你自己",
        "计算 123 + 456 * 789 等于多少？",
        "Python是什么？用一句话回答",
        "你好，请介绍一下你自己",  # 重复问题：应该命中缓存
        "计算 123 + 456 * 789 等于多少？",  # 重复问题：应该命中缓存
        "Python是什么？用一句话回答",  # 重复问题：应该命中缓存
        "什么是大语言模型？用一句话回答",  # 新问题：真实调用
    ]

    for i, question in enumerate(test_questions, 1):
        print(f"\n--- 第 {i} 次提问：{question} ---")
        try:
            result = agent.invoke({
                "messages": [{"role": "user", "content": question}]
            })
            # 提取最终回答
            final_answer = result["messages"][-1].content
            print(f"🤖 回答：{final_answer[:100]}..." if len(final_answer) > 100 else f"🤖 回答：{final_answer}")
        except Exception as e:
            print(f"❌ 调用出错：{str(e)}")

    # 输出最终统计
    print("\n" + "=" * 60)
    print("📊 缓存统计报告")
    print("=" * 60)
    for k, v in cache.get_stats().items():
        print(f"  {k}：{v}")
    print("=" * 60)


if __name__ == "__main__":
    run_demo()

LangChain Agent 缓存功能测试Demo

--- 第 1 次提问：你好，请介绍一下你自己 ---

🔍 缓存未命中，正在调用模型... (第 1 次真实调用)
✅ 模型调用完成，耗时：1.85秒
🤖 回答：你好！我是通义千问（Qwen），阿里巴巴集团旗下的超大规模语言模型。我能够回答问题、创作文字，比如写故事、写公文、写邮件、写剧本、逻辑推理、编程等等，还能表达观点，玩游戏等。如果你有任何问题或需要帮助...

--- 第 2 次提问：计算 123 + 456 * 789 等于多少？ ---

🔍 缓存未命中，正在调用模型... (第 2 次真实调用)
✅ 模型调用完成，耗时：0.87秒

🔍 缓存未命中，正在调用模型... (第 3 次真实调用)
✅ 模型调用完成，耗时：0.73秒
🤖 回答：123 + 456 × 789 = 359907

--- 第 3 次提问：Python是什么？用一句话回答 ---

🔍 缓存未命中，正在调用模型... (第 4 次真实调用)
✅ 模型调用完成，耗时：1.04秒
🤖 回答：Python是一种高级、通用、解释型编程语言，以简洁易读的语法和丰富的标准库著称。

--- 第 4 次提问：你好，请介绍一下你自己 ---

💾 缓存命中！累计命中 1 次
🤖 回答：你好！我是通义千问（Qwen），阿里巴巴集团旗下的超大规模语言模型。我能够回答问题、创作文字，比如写故事、写公文、写邮件、写剧本、逻辑推理、编程等等，还能表达观点，玩游戏等。如果你有任何问题或需要帮助...

--- 第 5 次提问：计算 123 + 456 * 789 等于多少？ ---

💾 缓存命中！累计命中 2 次

💾 缓存命中！累计命中 3 次
🤖 回答：123 + 456 × 789 = 359907

--- 第 6 次提问：Python是什么？用一句话回答 ---

💾 缓存命中！累计命中 4 次
🤖 回答：Python是一种高级、通用、解释型编程语言，以简洁易读的语法和丰富的标准库著称。

--- 第 7 次提问：什么是大语言模型？用一句话回答 ---

🔍 缓存未命中，正在调用模型... (第 5 次真实调用)
✅ 模型调用完成，耗时：0.92秒
🤖 回答：大语言模型是一种基于深度学习的自然语言处理模型，通过在海量文本数据上进行训练，能

场景3：修改系统提示

In [22]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain_core.messages import SystemMessage
from typing import Callable
import os


# ==========================================================
# 1. 加载环境变量 & 初始化模型
# ==========================================================
load_dotenv(override=True)

out_model = init_chat_model(
    model="qwen-plus",  # 可替换为 qwen-max / qwen-turbo / deepseek-chat 等
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    temperature=0,  # 温度设为0，相同输入输出更稳定，缓存效果更好
)

# ==========================================================
# 2. 定义动态上下文 Middleware
# ==========================================================
@wrap_model_call
def add_context(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    """动态添加上下文信息到系统提示"""
    from datetime import datetime

    # 获取当前时间
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S 星期%w")

    # 构建新的系统消息内容
    original_content = request.system_message.content if request.system_message else ""
    new_content = f"""{original_content}
        【运行时上下文 - 请在回答中利用这些信息】
        - 当前时间：{current_time}
        - 用户位置：中国 北京
        - 语言偏好：中文（请始终使用中文回答）
        """
    # 创建新的系统消息
    new_system_message = SystemMessage(content=new_content)
    # 用 override 替换请求中的 system_message
    modified_request = request.override(system_message=new_system_message)
    # 打印一下实际注入的系统提示，方便观察
    print(f"[Middleware] 已注入上下文，当前时间：{current_time}")
    return handler(modified_request)

# ==========================================================
# 3. 创建 Agent
# ==========================================================
agent = create_agent(
    model=out_model,
    middleware=[add_context],  # 注册我们的中间件
)

# ==========================================================
# 4. Demo 提问列表（用于验证上下文是否生效）
# ==========================================================
def run_demo():
    demo_questions = [
        "现在几点了？今天星期几？",
        "我现在在哪个国家哪个城市？",
        "请用三句话介绍一下你自己，并说明你会用什么语言回答我。",
        "如果我现在问你时间，你能准确回答吗？为什么？",
    ]

    print("=" * 60)
    print("  LangChain 动态上下文 Middleware Demo")
    print("=" * 60)

    for i, question in enumerate(demo_questions, 1):
        print(f"\n{'─' * 60}")
        print(f"❓ 第 {i} 次提问：{question}")
        print(f"{'─' * 60}")

        try:
            result = agent.invoke({"messages": [{"role": "user", "content": question}]})
            # 提取最后一条 AI 消息
            answer = result["messages"][-1].content
            print(f"🤖 回答：{answer}\n")
        except Exception as e:
            print(f"⚠️ 调用失败：{e}\n")

    print("=" * 60)
    print("  Demo 结束。观察：模型每次都能准确拿到当前时间和位置信息。")
    print("=" * 60)

if __name__ == "__main__":
    run_demo()

  LangChain 动态上下文 Middleware Demo

────────────────────────────────────────────────────────────
❓ 第 1 次提问：现在几点了？今天星期几？
────────────────────────────────────────────────────────────
[Middleware] 已注入上下文，当前时间：2026-10-10 22:12:30 星期6
🤖 回答：现在是 **2026年10月10日 22:12:30，星期六**（北京时间）。


────────────────────────────────────────────────────────────
❓ 第 2 次提问：我现在在哪个国家哪个城市？
────────────────────────────────────────────────────────────
[Middleware] 已注入上下文，当前时间：2026-10-10 22:12:31 星期6
🤖 回答：您当前位于**中国北京市**。


────────────────────────────────────────────────────────────
❓ 第 3 次提问：请用三句话介绍一下你自己，并说明你会用什么语言回答我。
────────────────────────────────────────────────────────────
[Middleware] 已注入上下文，当前时间：2026-10-10 22:12:32 星期6
🤖 回答：我是通义千问（Qwen），阿里巴巴研发的超大规模语言模型，具备多语言理解与生成能力，能回答问题、创作文字、编程、逻辑推理等。  
我当前已知你位于北京、偏好中文，因此**将始终使用中文**与你交流。  
我会结合实时上下文（如当前时间2026年10月10日）提供更准确、贴心的服务。


────────────────────────────────────────────────────────────
❓ 第 4 次提问：如果我现在问你时间，你能准确回答吗？为什么？
──────────────────────────────────────────────────────

### 2. wrap_tool_call

使用场景：用于监控、重试、修改工具执行。

2.1 基于装饰器实现

In [2]:
from langchain.chat_models import init_chat_model
from langchain.tools.tool_node import ToolCallRequest
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import HumanMessage, ToolMessage
from langchain.agents.middleware import wrap_tool_call
from dotenv import load_dotenv
from typing import Callable
from langgraph.types import Command
import os

load_dotenv(override=True)

out_model = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

@tool
def get_weather(city: str, is_forecast: bool) -> str:
    """
    获取当日特定城市的天气

    Args:
        city: 城市名称
        is_forecast: 是否包含明天的天气预报

    Returns:
        天气预报

    """
    res = f"{city}今天天气不错"
    if is_forecast:
        res += "\n明天天气不好，有暴雨。"
    return res

@wrap_tool_call
def wrap_tool_call_middleware(
        request: ToolCallRequest,
        handler: Callable[[ToolCallRequest], ToolMessage | Command],
) -> ToolMessage | Command:
    result = handler(request)
    print(f"原始参数：{request.tool_call['args']}")
    print(f"原始参数调用结果：{result}")

    request.tool_call["args"]["is_forecast"] = True
    result = handler(request)
    print(f"更新后的参数：{request.tool_call['args']}")
    print(f"更新参数调用结果：{result}")
    return result

agent = create_agent(
    model=out_model,
    tools=[get_weather],
    middleware=[wrap_tool_call_middleware],
)

resp = agent.invoke(
    {
        "messages": [HumanMessage("你好呀，今天深圳的天气怎么样？")]
    }
)

for msg in resp["messages"]:
    msg.pretty_print()

原始参数：{'city': '深圳', 'is_forecast': False}
原始参数调用结果：content='深圳今天天气不错' name='get_weather' tool_call_id='call_62215f7574aa425b8a4c67b2'
更新后的参数：{'city': '深圳', 'is_forecast': True}
更新参数调用结果：content='深圳今天天气不错\n明天天气不好，有暴雨。' name='get_weather' tool_call_id='call_62215f7574aa425b8a4c67b2'
================================ Human Message =================================

你好呀，今天深圳的天气怎么样？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_62215f7574aa425b8a4c67b2)
 Call ID: call_62215f7574aa425b8a4c67b2
  Args:
    city: 深圳
    is_forecast: True
================================= Tool Message =================================
Name: get_weather

深圳今天天气不错
明天天气不好，有暴雨。
================================== Ai Message ==================================

你好！今天深圳的天气很不错哦。不过明天会有暴雨，出门记得提前看好天气预报并做好防雨准备呀。


2.2 基于类实现

In [3]:
from langchain.chat_models import init_chat_model
from langchain.tools.tool_node import ToolCallRequest
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import HumanMessage, ToolMessage
from langchain.agents.middleware import AgentMiddleware
from dotenv import load_dotenv
from typing import Callable,Any
from langgraph.types import Command
import os

load_dotenv(override=True)

out_model = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

@tool
def get_weather(city: str, is_forecast: bool) -> str:
    """
    获取当日特定城市的天气

    Args:
        city: 城市名称
        is_forecast: 是否包含明天的天气预报

    Returns:
        天气预报

    """
    res = f"{city}今天天气不错"
    if is_forecast:
        res += "\n明天天气不好，有暴雨。"
    return res

# 继承 AgentMiddleware类，并且重新它的核心方法
class MyWrapToolCallMiddleware(AgentMiddleware):
    def wrap_tool_call(
        self,
        request: ToolCallRequest,
        handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
    ) -> ToolMessage | Command[Any]:
        result = handler(request)
        print(f"原始参数：{request.tool_call['args']}")
        print(f"原始参数调用结果：{result}")

        request.tool_call["args"]["is_forecast"] = True
        result = handler(request)
        print(f"更新后的参数：{request.tool_call['args']}")
        print(f"更新参数调用结果：{result}")
        return result

agent = create_agent(
    model=out_model,
    tools=[get_weather],
    middleware=[MyWrapToolCallMiddleware()],
)

resp = agent.invoke(
    {
        "messages": [HumanMessage("你好呀，今天深圳的天气怎么样？")]
    }
)

for msg in resp["messages"]:
    msg.pretty_print()

原始参数：{'city': '深圳', 'is_forecast': False}
原始参数调用结果：content='深圳今天天气不错' name='get_weather' tool_call_id='call_b48778d8eaa74f15969c7154'
更新后的参数：{'city': '深圳', 'is_forecast': True}
更新参数调用结果：content='深圳今天天气不错\n明天天气不好，有暴雨。' name='get_weather' tool_call_id='call_b48778d8eaa74f15969c7154'
================================ Human Message =================================

你好呀，今天深圳的天气怎么样？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_b48778d8eaa74f15969c7154)
 Call ID: call_b48778d8eaa74f15969c7154
  Args:
    city: 深圳
    is_forecast: True
================================= Tool Message =================================
Name: get_weather

深圳今天天气不错
明天天气不好，有暴雨。
================================== Ai Message ==================================

你好呀！今天深圳的天气不错哦。不过要提醒你一下，明天会有暴雨，记得提前做好出行准备。


### 装饰器和类的选择
**情况1：中间件只用一个钩子函数，推荐用装饰器，需要多个钩子函数推荐类写法**
- 当一个中间件**只需要实现一个钩子函数**时，直接使用装饰器最简单。
- 当一个中间件**需要实现多个钩子函数**时，类写法更合适。

装饰器也不是不能实现，多数情况下可以像下面的示例里那样通过**工厂函数返回多个装饰器函数**来完成；但这种方式本质上是把一个“逻辑上属于同一个中间件”的行为拆成多个独立函数，再由外部统一组装，因此**不如类写法自然、集中、清晰**。

3.1 使用装饰器

In [4]:
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import before_model, after_model, AgentState
from langchain.messages import HumanMessage
from langgraph.runtime import Runtime
from loguru import logger
from typing import Any
from dotenv import load_dotenv
from langchain.agents import create_agent
import os

load_dotenv(override=True)

out_model = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

# 将多个装饰器放在一起
def create_audit_middleware(logger):
    @before_model
    def before_log(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        logger.info("调用模型前消息数量：{}", len(state["messages"]))
        return None

    @after_model
    def after_log(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        logger.info("调用模型后消息数量：{}", len(state["messages"]))
        return None

    return [before_log, after_log]

agent = create_agent(
    model= out_model,
    middleware=[*create_audit_middleware(logger= logger)], #多个hook的调用
)

resp = agent.invoke({
    "messages": [
        HumanMessage("你好呀！")
    ]
})

for msg in resp["messages"]:
    msg.pretty_print()

2026-10-11 20:11:25.001 | INFO     | __main__:before_log:24 - 调用模型前消息数量：1
2026-10-11 20:11:28.652 | INFO     | __main__:after_log:29 - 调用模型后消息数量：2


================================ Human Message =================================

你好呀！
================================== Ai Message ==================================

你好呀！很高兴和你聊天，今天有什么我可以帮你的吗？


3.2 使用类

In [5]:
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import AgentMiddleware, AgentState
from langchain.messages import HumanMessage
from langgraph.runtime import Runtime
from loguru import logger
from typing import Any
from dotenv import load_dotenv
from langchain.agents import create_agent
import os

load_dotenv(override=True)

out_model = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

class CreateAuditMiddleware(AgentMiddleware):
    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        logger.info("调用模型前消息数量：{}", len(state["messages"]))
        return None
    def after_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        logger.info("调用模型后消息数量：{}", len(state["messages"]))
        return None

agent = create_agent(
    model=out_model,
    middleware=[CreateAuditMiddleware()],
)

resp = agent.invoke({
    "messages": [
        HumanMessage("你好呀！")
    ]
})

for msg in resp["messages"]:
    msg.pretty_print()

2026-10-11 20:19:05.047 | INFO     | __main__:before_model:22 - 调用模型前消息数量：1
2026-10-11 20:19:08.497 | INFO     | __main__:after_model:25 - 调用模型后消息数量：2


================================ Human Message =================================

你好呀！
================================== Ai Message ==================================

你好呀！很高兴见到你。😊

请问今天有什么我可以帮你的吗？无论是解答疑惑、协助工作，还是随便聊聊天，我都随时待命哦！


结合上面的两个示例，可以得出结论：
- 装饰器写法适合把单个 hook 快速挂到 agent 生命周期的某个节点上；
- 类写法更适合把多个 hook 组织为一个完整的中间件组件；
- 当中间件同时涉及 before_model 、 after_model 等多个钩子时，虽然装饰器工厂也能实现，但类写法在结构表达、配置归属、可维护性上更好。

总结：单钩子场景下，装饰器即可；多钩子场景下，类不是唯一可行方案，但通常是更自然、更推荐的实现方式。

**情况2：复杂配置推荐用类实现**

In [6]:
from langchain.agents.middleware import AgentMiddleware, AgentState, before_model
from langgraph.runtime import Runtime
from loguru import logger
from typing import Any

#基于类的方法
class AuditMiddleware(AgentMiddleware):
    def __init__(self, logger, threshold: int, middleware_name: str) -> None:
        self.logger = logger
        self.threshold = threshold
        self.middleware_name = middleware_name

    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        self.logger.info("current name: {}， threshold: {}", self.middleware_name, self.threshold)
        return None

#基于装饰器的方法，传参要通过闭包完成
def create_audit_middleware(logger, threshold: int, middleware_name: str):
    @before_model
    def audit_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        logger.info("current name: {}， threshold: {}", middleware_name, threshold)
        return None
    return audit_middleware

class_middle = [
    AuditMiddleware(logger=logger, threshold=5, middleware_name="short limit"),
    AuditMiddleware(logger=logger, threshold=50, middleware_name="long limit"),
]

decorate_middle = [
    create_audit_middleware(logger=logger, threshold=5, middleware_name="short limit"),
    create_audit_middleware(logger=logger, threshold=50, middleware_name="long limit"),
]

print("=" * 30, " ---> class风格的中间件 <--- ", "=" * 30)
for mw in class_middle:
    print(type(mw))
    print(mw.__dict__)

print("=" * 30, " ---> decorate风格的中间件 <--- ", "=" * 30)
for mw in decorate_middle:
    print(type(mw))
    print(mw.__dict__)

==============================  ---> class风格的中间件 <---  ==============================
<class '__main__.AuditMiddleware'>
{'logger': <loguru.logger handlers=[(id=0, level=10, sink=stderr)]>, 'threshold': 5, 'middleware_name': 'short limit'}
<class '__main__.AuditMiddleware'>
{'logger': <loguru.logger handlers=[(id=0, level=10, sink=stderr)]>, 'threshold': 50, 'middleware_name': 'long limit'}
==============================  ---> decorate风格的中间件 <---  ==============================
<class 'langchain.agents.middleware.types.audit_middleware'>
{}
<class 'langchain.agents.middleware.types.audit_middleware'>
{}


情况3：跨项目复用推荐用类写法

如果希望中间件成为一个可实例化、可封装、可测试的组件，类写法更加合适，因为这些本就是类擅长的场景，装饰器的闭包也能实现，但使用不友好。